準備0. OS,GPU,Tensorflowの確認

In [ ]:
!cat /etc/os-release

PRETTY_NAME="Ubuntu 22.04.4 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.4 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


T4, L4は動作確認済み

In [ ]:
!nvidia-smi

Thu Apr 17 11:58:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import tensorflow as tf
tf.test.gpu_device_name()

'/device:GPU:0'

準備1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


準備1-1. ドライブを再読み込み

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


準備3. Donkey Car5.1.0をダウンロード

In [ ]:
!git clone https://github.com/autorope/donkeycar
%cd ./donkeycar
!git checkout 5.1.0

Cloning into 'donkeycar'...
remote: Enumerating objects: 16443, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 16443 (delta 0), reused 0 (delta 0), pack-reused 16440 (from 2)
Receiving objects: 100% (16443/16443), 90.13 MiB | 20.48 MiB/s, done.
Resolving deltas: 100% (10916/10916), done.
/content/donkeycar
Note: switching to '5.1.0'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at bd5521f9 Set release version


準備4. Donkey Car5.1.0をインストール

In [ ]:
%cd /content/donkeycar
!pip install -e .[pc] --quiet

/content/donkeycar
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for donkeycar (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.15.1 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.15.1 which is incompatible.
jax 0.5.2 requires ml_dtypes>=0.4.0, but you have ml-dtypes 0.3.2 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.15.1 which is incompatible.


準備5. Donkey Car5.1.0とColabGPUに対応したTensorflowのインストール

In [ ]:
!pip install tensorflow==2.18.0 --quiet

In [ ]:
!pip install albumentations==1.3.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.7
    Uninstalling albumentations-2.0.7:
      Successfully uninstalled albumentations-2.0.7


準備5.1. ハイパーパラメータ用Optunaをとモデル名のためのライブラリをインストール

In [ ]:
!pip install optuna pytz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 25.9 MB/s eta 0:00:00


準備6.mycarを作成

In [ ]:
!donkey createcar --path /content/mycar

________             ______                   _________              
___  __ \_______________  /___________  __    __  ____/_____ ________
__  / / /  __ \_  __ \_  //_/  _ \_  / / /    _  /    _  __ `/_  ___/
_  /_/ // /_/ /  / / /  ,<  /  __/  /_/ /     / /___  / /_/ /_  /    
/_____/ \____//_/ /_//_/|_| \___/_\__, /      \____/  \__,_/ /_/     
                                 /____/                              

using donkey v5.1.0 ...
Creating car folder: /content/mycar
making dir  /content/mycar
Creating data & model folders.
making dir  /content/mycar/models
making dir  /content/mycar/data
making dir  /content/mycar/logs
Copying car application template: complete
Copying car config defaults. Adjust these before starting your car.
Copying train script. Adjust these before starting your car.
Copying calibrate script. Adjust these before starting your car.
Copying my car config overrides
Donkey setup complete.


準備7. GPU、ハイパーパラメータチューニングに対応した改造ファイルを組み込む

In [ ]:
!cp /content/drive/MyDrive/keras.py /content/donkeycar/donkeycar/parts/keras.py
!cp /content/drive/MyDrive/interpreter.py /content/donkeycar/donkeycar/parts/interpreter.py
!cp /content/drive/MyDrive/train.py /content/mycar/train.py
!cp /content/drive/MyDrive/myconfig.py /content/mycar/myconfig.py
!cp /content/drive/MyDrive/manage.py /content/mycar/manage.py

ここまではランタイム起動から１回だけでOK

////////////////////////////////////////////////////////////////////////////////

1. 教師データの解凍,移動

In [ ]:
%cd /content/mycar
!unzip data.zip

ストリーミング出力は最後の 5000 行に切り捨てられました。
  inflating: 4_15/images/10747_cam_image_array_.jpg  
  inflating: 4_15/images/10799_cam_image_array_.jpg  
  inflating: 4_15/images/7001_cam_image_array_.jpg  
  inflating: 4_15/images/6503_cam_image_array_.jpg  
  inflating: 4_15/images/11418_cam_image_array_.jpg  
  inflating: 4_15/images/1500_cam_image_array_.jpg  
  inflating: 4_15/images/6271_cam_image_array_.jpg  
  inflating: 4_15/images/3674_cam_image_array_.jpg  
  inflating: 4_15/images/2358_cam_image_array_.jpg  
  inflating: 4_15/images/2456_cam_image_array_.jpg  
  inflating: 4_15/images/7642_cam_image_array_.jpg  
  inflating: 4_15/images/1566_cam_image_array_.jpg  
  inflating: 4_15/images/1185_cam_image_array_.jpg  
  inflating: 4_15/images/9386_cam_image_array_.jpg  
  inflating: 4_15/images/11229_cam_image_array_.jpg  
  inflating: 4_15/images/2664_cam_image_array_.jpg  
  inflating: 4_15/images/778_cam_image_array_.jpg  
  inflating: 4_15/images/8333_cam_image_array_.jpg  
  inflating

In [ ]:
!cp /content/data /content/mycar

2. 学習開始

In [ ]:
#train.py  --tub data  --model models/model.keras --transfer=<transfer model path>

mycarの中にtfliteモデルは保存される
日時、モデルの種類が名前に
↓一般的な学習

In [ ]:
%cd /content/mycar
!donkey train  --tub /content/mycar/4_15  --model /content/mycar/models/model.keras --type linear

/content/mycar
________             ______                   _________              
___  __ \_______________  /___________  __    __  ____/_____ ________
__  / / /  __ \_  __ \_  //_/  _ \_  / / /    _  /    _  __ `/_  ___/
_  /_/ // /_/ /  / / /  ,<  /  __/  /_/ /     / /___  / /_/ /_  /    
/_____/ \____//_/ /_//_/|_| \___/_\__, /      \____/  \__,_/ /_/     
                                 /____/                              

using donkey v5.1.0 ...
INFO:donkeycar.config:loading config file: ./config.py
INFO:donkeycar.config:loading personal config over-rides from ./myconfig.py
2025-05-28 08:57:11.447335: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748422631.467509    7219 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748422631.4

↓ハイパーパラメータチューニング付き(tryが試行回数)

In [ ]:
%cd /content/mycar
!python manage.py tune --tub 4_15 --model /content/mycar/models/ --type linear --try 15

/content/mycar
________             ______                   _________              
___  __ \_______________  /___________  __    __  ____/_____ ________
__  / / /  __ \_  __ \_  //_/  _ \_  / / /    _  /    _  __ `/_  ___/
_  /_/ // /_/ /  / / /  ,<  /  __/  /_/ /     / /___  / /_/ /_  /    
/_____/ \____//_/ /_//_/|_| \___/_\__, /      \____/  \__,_/ /_/     
                                 /____/                              

using donkey v5.1.0 ...
INFO:donkeycar.config:loading config file: /content/mycar/config.py
INFO:donkeycar.config:loading personal config over-rides from myconfig.py
2025-05-28 08:49:18.181432: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748422158.202488    4920 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1

In [ ]:
from google.colab import files
files.download('/content/mycar/models/model03_2_3.tflite')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>